# NumPy: The Bridge to Data Science and ML

A quick refresher on the key terms before we get into the connections:

- **NumPy** (short for *Numerical Python*): a Python library for working with large arrays of numbers and doing math on them quickly.
- **Array**: a grid of numbers, all of the same data type. Can be 1D (a list), 2D (a table), or higher-dimensional.
- **Data Science**: the practice of pulling insights out of data using a mix of statistics, programming, and visualization.
- **Machine Learning (ML)**: teaching a computer to find patterns in data so it can make predictions on new, unseen data.

In this notebook we coveer how NumPy sits underneath nearly every popular tool in the data and ML world, and we build a small ML workflow from scratch to see this in action.

Topics covered:
1. How NumPy connects to pandas DataFrames
2. How NumPy underlies scikit-learn (and every other ML library)
3. Images as NumPy arrays
4. A small ML workflow using only NumPy
5. When to reach for NumPy vs pandas vs frameworks
6. Where to go from here

In [1]:
import numpy as np
np.random.seed(42)
print("NumPy version:", np.__version__)

NumPy version: 1.26.4


## NumPy and Pandas

A few terms before we start:
- **Pandas**: a Python library for working with labeled, tabular data.
- **DataFrame**: pandas' main data structure. Think of it as a spreadsheet, with named columns and indexed rows.
- **Series**: a single column of a DataFrame.

Pandas DataFrames are built on top of NumPy arrays. Every numeric column you see in a DataFrame is a NumPy array underneath. So when you're learning NumPy, you're really learning the engine that powers pandas.

In [2]:
# Simulate a DataFrame-like dataset using NumPy
# Imagine: student records with (id, age, score, hours_studied)

n = 8
ids        = np.arange(1, n + 1)
ages       = np.array([20, 22, 19, 21, 23, 20, 22, 21])
scores     = np.array([85, 72, 90, 68, 78, 92, 80, 75])
hours      = np.array([15, 8, 20, 5, 12, 25, 10, 7])

print("IDs:    ", ids)
print("Ages:   ", ages)
print("Scores: ", scores)
print("Hours:  ", hours)

IDs:     [1 2 3 4 5 6 7 8]
Ages:    [20 22 19 21 23 20 22 21]
Scores:  [85 72 90 68 78 92 80 75]
Hours:   [15  8 20  5 12 25 10  7]


In [3]:
# Combine into a 2D matrix (rows = students, cols = features)
data = np.column_stack([ids, ages, scores, hours])
print("Combined data:")
print(data)
print(f"Shape: {data.shape}")

Combined data:
[[ 1 20 85 15]
 [ 2 22 72  8]
 [ 3 19 90 20]
 [ 4 21 68  5]
 [ 5 23 78 12]
 [ 6 20 92 25]
 [ 7 22 80 10]
 [ 8 21 75  7]]
Shape: (8, 4)


### The same operations, in pandas and in NumPy

If you've used pandas before, the calls below are the NumPy equivalents of common DataFrame operations.

In [4]:
# pandas: df['score'].mean()
print(f"Mean score:  {scores.mean():.2f}")

# pandas: df[df['score'] > 80]
mask = scores > 80
print(f"\nHigh scorers (score > 80):")
print(data[mask])

# pandas: df.sort_values('score')
sorted_idx = np.argsort(scores)
print(f"\nSorted by score (ascending):")
print(data[sorted_idx])

# pandas: df['score'].corr(df['hours'])
corr = np.corrcoef(scores, hours)[0, 1]
print(f"\nCorrelation between scores and hours: {corr:.4f}")

Mean score:  80.00

High scorers (score > 80):
[[ 1 20 85 15]
 [ 3 19 90 20]
 [ 6 20 92 25]]

Sorted by score (ascending):
[[ 4 21 68  5]
 [ 2 22 72  8]
 [ 8 21 75  7]
 [ 5 23 78 12]
 [ 7 22 80 10]
 [ 1 20 85 15]
 [ 3 19 90 20]
 [ 6 20 92 25]]

Correlation between scores and hours: 0.9571


Pandas is a friendlier interface when your data has mixed types and column labels. The math underneath, though, is still NumPy. For raw numerical work, or when speed matters more than convenience, working directly with NumPy is often the better fit.

## NumPy is the common language of ML libraries

A quick rundown of the libraries mentioned below:
- **scikit-learn**: the standard Python library for classical machine learning (regression, classification, clustering, and so on).
- **PyTorch**: a deep learning library from Meta, widely used in research and increasingly in production.
- **TensorFlow**: a deep learning library from Google.
- **XGBoost / LightGBM**: libraries for gradient boosting, a technique that performs especially well on tabular data and frequently shows up in Kaggle competition winners.
- **statsmodels**: a library focused on statistics and econometrics.
- **Tensor**: the deep learning equivalent of a NumPy array. PyTorch and TensorFlow operate on tensors, but tensors behave very much like NumPy arrays.

All of these libraries accept NumPy arrays as input. scikit-learn, XGBoost, LightGBM, and statsmodels use them directly. PyTorch and TensorFlow convert them into tensors internally.

The shape conventions are shared across the ecosystem too:
- **X (features)**: a 2D array with shape `(n_samples, n_features)`
- **y (labels)**: a 1D array with shape `(n_samples,)`

Once this pattern clicks, switching between ML libraries becomes much less intimidating.

In [5]:
# A typical ML data layout — even without sklearn
# 100 samples, 4 features each
n_samples = 100
n_features = 4

X = np.random.randn(n_samples, n_features)   # features
y = np.random.randint(0, 2, n_samples)        # binary labels (0 or 1)

print(f"X shape: {X.shape}   ← (n_samples, n_features)")
print(f"y shape: {y.shape}   ← (n_samples,)")

# This is the EXACT format every sklearn estimator expects:
#   model.fit(X, y)
#   predictions = model.predict(X_new)

X shape: (100, 4)   ← (n_samples, n_features)
y shape: (100,)   ← (n_samples,)


## Images are NumPy arrays

- **Pixel**: a single dot in an image. Each pixel has values that represent its color or brightness.
- **Grayscale image**: an image where each pixel is one number, usually between 0 (black) and 255 (white).
- **RGB image**: a color image where each pixel has three numbers: one for Red, one for Green, and one for Blue.
- **Channel**: one of the layers in an image. A grayscale image has 1 channel; an RGB image has 3.

This is the idea that unlocks computer vision: an image is just a NumPy array. Grayscale images have shape `(height, width)`. Color images have shape `(height, width, 3)`. Once you accept that, "image processing" becomes a special case of array manipulation.

In [6]:
# Create a simple 5x5 grayscale "image"
# Values 0-255 represent pixel brightness
img = np.array([
    [  0,  50, 100, 150, 200],
    [ 50, 100, 150, 200, 250],
    [100, 150, 200, 250, 255],
    [150, 200, 250, 255, 200],
    [200, 250, 255, 200, 150]
], dtype=np.uint8)

print("A 5x5 grayscale image:")
print(img)
print(f"\nShape: {img.shape}, dtype: {img.dtype}")

A 5x5 grayscale image:
[[  0  50 100 150 200]
 [ 50 100 150 200 250]
 [100 150 200 250 255]
 [150 200 250 255 200]
 [200 250 255 200 150]]

Shape: (5, 5), dtype: uint8


In [7]:
# Common image operations are just array operations

# Invert the image (255 - pixel)
inverted = 255 - img
print("Inverted:")
print(inverted)

# Increase brightness (add a constant, clip to valid range)
brighter = np.clip(img.astype(int) + 50, 0, 255).astype(np.uint8)
print("\nBrightened by 50:")
print(brighter)

# Threshold (binary mask)
mask = (img > 150).astype(np.uint8) * 255
print("\nThreshold > 150:")
print(mask)

Inverted:
[[255 205 155 105  55]
 [205 155 105  55   5]
 [155 105  55   5   0]
 [105  55   5   0  55]
 [ 55   5   0  55 105]]

Brightened by 50:
[[ 50 100 150 200 250]
 [100 150 200 250 255]
 [150 200 250 255 255]
 [200 250 255 255 250]
 [250 255 255 250 200]]

Threshold > 150:
[[  0   0   0   0 255]
 [  0   0   0 255 255]
 [  0   0 255 255 255]
 [  0 255 255 255 255]
 [255 255 255 255   0]]


In [8]:
# A color image — shape (height, width, 3) for RGB
# Create a 4x4 color "image"
color_img = np.zeros((4, 4, 3), dtype=np.uint8)

# Set the red channel of the top-left 2x2 region to max
color_img[:2, :2, 0] = 255

# Set the green channel of the bottom-right 2x2 to max
color_img[2:, 2:, 1] = 255

print("Shape:", color_img.shape, "(height, width, channels)")
print("\nRed channel:")
print(color_img[:, :, 0])
print("\nGreen channel:")
print(color_img[:, :, 1])
print("\nBlue channel:")
print(color_img[:, :, 2])

Shape: (4, 4, 3) (height, width, channels)

Red channel:
[[255 255   0   0]
 [255 255   0   0]
 [  0   0   0   0]
 [  0   0   0   0]]

Green channel:
[[  0   0   0   0]
 [  0   0   0   0]
 [  0   0 255 255]
 [  0   0 255 255]]

Blue channel:
[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]


## When to use NumPy and when to use something else

NumPy isn't always the right answer. A rough guide:

| Scenario | Best tool |
|---|---|
| Pure numerical math | NumPy |
| Linear algebra, statistics | NumPy + SciPy |
| Tabular data with mixed types and labels | pandas (built on NumPy) |
| Classical machine learning | scikit-learn |
| Deep learning | PyTorch or TensorFlow |
| GPU-accelerated math | CuPy (NumPy on GPU), PyTorch tensors |
| Data bigger than RAM | Dask, Polars, Vaex |
| Image processing | OpenCV, PIL, scikit-image (all NumPy-based) |

Quick notes on the less familiar ones:
- **SciPy**: built on NumPy, adds more advanced math (optimization, signal processing, advanced statistics).
- **CuPy**: NumPy's API, but the arrays live on a GPU instead of in normal memory.
- **Dask**: lets you work with arrays and DataFrames that don't fit in memory by splitting them into chunks.
- **Polars**: a fast DataFrame library written in Rust. Similar feel to pandas, built for speed.
- **Vaex**: another out-of-core DataFrame library, geared toward very large datasets.

The nice part: if you're solid on NumPy, all of these feel familiar. NumPy is the foundation they either build on or borrow from.